# 📓 Semana 4 · Dia 5 — Camada Ouro: agregados de negócio

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (medallion, BI) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Tabelas Ouro criadas + dashboard conectado |

---


## 📖 Teoria — O papel do Ouro

O **Ouro** é o que analistas, BI e modelos consomem: denormalizado, agregado e estável. 'Denormalizado' = joins já resolvidos, KPI direto.

Regras: poucas tabelas, colunas claras, sem lixo, sempre atualizado pelo pipeline.


### 💻 Na prática — Criando os agregados

Crie as 3 tabelas Ouro do projeto a partir da Prata.


In [ ]:
# vendas_por_dia (série temporal para BI e previsão)
vendas_por_dia = (spark.table("workspace.prata.fato_vendas")
    .groupBy("sk_tempo")
    .agg(s("receita").alias("receita_total"),
         count("*").alias("n_vendas"),
         countDistinct("InvoiceNo").alias("n_notas"))
    .join(spark.table("workspace.prata.dim_tempo").select("sk_tempo", "data_venda"), "sk_tempo")
    .select("data_venda", "receita_total", "n_vendas", "n_notas")
    .orderBy("data_venda"))
vendas_por_dia.show(5)

In [ ]:
# receita_por_pais
receita_por_pais = (spark.table("workspace.prata.fato_vendas")
    .groupBy("Country")
    .agg(s("receita").alias("receita_total"), count("*").alias("n_vendas"))
    .orderBy(col("receita_total").desc()))
receita_por_pais.show(5)

In [ ]:
# top_produtos
top_produtos = (spark.table("workspace.prata.fato_vendas")
    .groupBy("sk_produto")
    .agg(s("receita").alias("receita_total"), count("*").alias("n_vendas"))
    .join(spark.table("workspace.prata.dim_produto").select("sk_produto", "StockCode", "Description"), "sk_produto")
    .orderBy(col("receita_total").desc())
    .limit(20))
top_produtos.show(5, truncate=False)

In [ ]:
# Gravar o Ouro
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ouro")
vendas_por_dia.write.mode("overwrite").saveAsTable("workspace.ouro.vendas_por_dia")
receita_por_pais.write.mode("overwrite").saveAsTable("workspace.ouro.receita_por_pais")
top_produtos.write.mode("overwrite").saveAsTable("workspace.ouro.top_produtos")
print("Camada Ouro criada!")

### 💻 Na prática — Dashboard conectado ao Ouro

Crie visualizações no Databricks SQL sobre o Ouro.


In [ ]:
%sql
-- Query para o dashboard
SELECT data_venda, receita_total
FROM workspace.ouro.vendas_por_dia
ORDER BY data_venda

> 🎯 **Dica de prova**: A Ouro é **denormalizada e agregada** para BI/IA — nunca dados brutos. Pergunta típica: 'onde coloco uma view de KPI?' → Ouro.


## 🎯 Exercícios de fixação

**1.** Crie uma tabela Ouro `vendas_por_dia_pais` (data × país × receita).

**2.** Qual camada um modelo de ML deve consumir? Por quê?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** vendas_por_dia_pais

```python
(spark.table('workspace.prata.fato_vendas').groupBy('sk_tempo','Country').agg(sum('receita').alias('receita')).join(spark.table('workspace.prata.dim_tempo'),'sk_tempo').select('data_venda','Country','receita').write.mode('overwrite').saveAsTable('workspace.ouro.vendas_por_dia_pais'))
```

**2.** ML no Ouro

Ouro: agregados limpos e estáveis → treinar previsão de receita. (Para features granulares, usa-se Feature Engineering sobre Prata/Ouro — Semana 10.)



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*